# DDPM Cancer Detection - Kaggle训练

这个Notebook用于在Kaggle上训练DDPM癌症检测图像生成模型。

## 步骤：
1. 设置环境
2. 准备数据集
3. 训练模型
4. 生成图像

## 1. 设置环境

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
!pip install denoising-diffusion-pytorch einops ema-pytorch accelerate

## 2. 准备数据集

In [ ]:
import os
import shutil
import pandas as pd
from PIL import Image
from tqdm import tqdm

KAGGLE_DATA_DIR = '/kaggle/input/histopathologic-cancer-detection'
WORKING_DIR = '/kaggle/working'
DATA_DIR = os.path.join(WORKING_DIR, 'data')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'positive'), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'negative'), exist_ok=True)

In [ ]:
def prepare_dataset(num_samples=None, image_size=96):
    """
    准备数据集，从Kaggle数据集复制并分类
    """
    df = pd.read_csv(os.path.join(KAGGLE_DATA_DIR, 'train_labels.csv'))
    
    if num_samples:
        df = df.sample(n=num_samples, random_state=42)
    
    print(f"总样本数: {len(df)}")
    print(f"阳性样本 (label=0): {len(df[df['label'] == 0])}")
    print(f"阴性样本 (label=1): {len(df[df['label'] == 1])}")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="处理图像"):
        image_id = row['id']
        label = row['label']
        
        src_path = os.path.join(KAGGLE_DATA_DIR, 'train', f"{image_id}.tif")
        
        if label == 0:
            dst_path = os.path.join(DATA_DIR, 'positive', f"{image_id}.png")
        else:
            dst_path = os.path.join(DATA_DIR, 'negative', f"{image_id}.png")
        
        try:
            img = Image.open(src_path)
            if img.size != (image_size, image_size):
                img = img.resize((image_size, image_size), Image.BILINEAR)
            img.save(dst_path, 'PNG')
        except Exception as e:
            print(f"处理图像 {image_id} 时出错: {e}")
    
    print("数据集准备完成!")

prepare_dataset(num_samples=5000, image_size=96)

## 3. 训练模型

In [ ]:
from denoising_diffusion_pytorch import Unet, GaussianDiffusion, Trainer
import torch

model = Unet(
    dim = 64,
    dim_mults = (1, 2, 4, 8),
    channels = 3,
    flash_attn = False
)

diffusion = GaussianDiffusion(
    model,
    image_size = 96,
    timesteps = 1000,
    sampling_timesteps = 250
)

trainer = Trainer(
    diffusion,
    os.path.join(DATA_DIR, 'positive'),
    train_batch_size = 32,
    train_lr = 8e-5,
    train_num_steps = 10000,
    gradient_accumulate_every = 2,
    ema_decay = 0.995,
    amp = True,
    results_folder = os.path.join(WORKING_DIR, 'results'),
    save_and_sample_every = 1000
)

trainer.train()

## 4. 生成图像

In [ ]:
from denoising_diffusion_pytorch import Unet, GaussianDiffusion
import torch
from PIL import Image
import os

def generate_images(model_path, num_images=16, image_size=96):
    """
    生成图像
    """
    model = Unet(
        dim = 64,
        dim_mults = (1, 2, 4, 8),
        channels = 3,
        flash_attn = False
    )
    
    diffusion = GaussianDiffusion(
        model,
        image_size = image_size,
        timesteps = 1000,
        sampling_timesteps = 250
    )
    
    diffusion.load(model_path)
    diffusion = diffusion.cuda()
    
    generated_images = diffusion.sample(batch_size = num_images)
    
    return generated_images

model_path = os.path.join(WORKING_DIR, 'results', 'model-10.pt')
if os.path.exists(model_path):
    images = generate_images(model_path, num_images=16)
    
    for i, img in enumerate(images):
        img_pil = Image.fromarray((img.permute(1, 2, 0).cpu().numpy() * 255).astype('uint8'))
        img_pil.save(os.path.join(WORKING_DIR, f'generated_{i}.png'))
        display(img_pil)
else:
    print(f"模型文件不存在: {model_path}")
    print("请先完成训练!")

## 5. 下载结果

In [ ]:
import os
from IPython.display import FileLink

results_dir = os.path.join(WORKING_DIR, 'results')
if os.path.exists(results_dir):
    print("可用的结果文件:")
    for f in os.listdir(results_dir):
        print(f"  - {f}")
        if f.endswith('.pt'):
            display(FileLink(os.path.join(results_dir, f)))
else:
    print("结果目录不存在")